# Experimento de Comparação de Janelas de Treinamento

**Projeto:** Renewable Energy MLOps — Wind Power Forecasting  
**Status:** preparado para execução  
**Branch:** `experiment/training-window-comparison`  
**Snapshot temporal:** congelado  
**Arquitetura:** `LGBM + XGBoost + Random Forest`

Este notebook documenta um experimento controlado para avaliar quanto
histórico deve ser utilizado no treinamento do Temporal Stacking.

A variável experimental é **somente a janela temporal de treinamento**:

```text
EXPANDING
└── todo o histórico canônico disponível antes do OOT

ROLLING 24M
└── últimos 24 meses antes do OOT

ROLLING 12M
└── últimos 12 meses antes do OOT
```

O notebook anterior de simplificação do ensemble permanece congelado.
Este experimento não altera a arquitetura do modelo e não executa o
Quality Gate operacional.


## 1. Objetivo

O objetivo é verificar se restringir o treinamento aos regimes mais
recentes melhora ou preserva a capacidade de generalização em relação
à expanding window.

A expanding window não é tratada como proteção automática contra
data drift. À medida que o processo gerador dos dados muda, histórico
mais antigo pode deixar de representar adequadamente o regime atual.

O benchmark mantém fixos:

- o mesmo snapshot Gold;
- o mesmo OOT de setembro/2026;
- as mesmas 9 features;
- a mesma arquitetura `LGBM + XGB + RF`;
- o mesmo `TimeSeriesSplit` externo;
- o mesmo `TimeSeriesSplit` interno do Temporal Stacking;
- 20 trials Optuna por tratamento;
- nMAE (%) em MW como métrica primária de otimização.

Somente o **limite inferior do TRAIN** muda entre os tratamentos.


## 2. Hipóteses

### Pergunta experimental

Quanto histórico deve ser utilizado para treinar o Temporal Stacking
quando o futuro avaliado é setembro/2026?

### Tratamentos

```text
EXPANDING
2024-03-21 00:00 → 2026-09-01 02:00

ROLLING 24M
2024-09-01 03:00 → 2026-09-01 02:00

ROLLING 12M
2025-09-01 03:00 → 2026-09-01 02:00

OOT COMUM
2026-09-01 03:00 → 2026-10-01 02:00
```

### Hipótese de trabalho

Uma janela recente pode reduzir a influência de regimes históricos
menos representativos do estado atual do sistema.

O experimento não assume antecipadamente que uma rolling window será
superior. A expanding window permanece como baseline.


## 3. Contrato experimental

```text
Snapshot Gold congelado
        ↓
split_training_window_snapshot()
        │
        ├── None → expanding
        ├── 24   → rolling 24m
        └── 12   → rolling 12m
        │
        └───────────────┐
                        ↓
                   TRAIN distinto
                        +
                    OOT idêntico
                        ↓
                 Optuna idêntico
                        ↓
             LGBM + XGB + RF
                        ↓
                     MLflow
```

O notebook reutiliza a função oficial de split do `training_flow`.
Nenhuma segunda implementação de corte temporal é criada aqui.

O `continuous_training_pipeline()` não é chamado, portanto
`evaluate_and_promote()` também não é executado.


In [1]:
import importlib
import inspect
import json

import pandas as pd
from mlflow.tracking import MlflowClient

import energy_mlops.models.optimize_stacking_ensemble as optimization_module
import energy_mlops.models.train_stacking_ensemble as training_module
from energy_mlops.config import settings
from energy_mlops.data.feature_utils import get_model_feature_columns
from energy_mlops.data.snapshot_validation import audit_gold_snapshot
from energy_mlops.models.ensemble_config import validate_enabled_estimators
from energy_mlops.pipelines.training_flow import (
    build_training_dataset_labels,
    persist_training_datasets_task,
    split_training_window_snapshot,
)

optimization_module = importlib.reload(
    optimization_module
)

training_module = importlib.reload(
    training_module
)

print("✅ Módulos recarregados.")

print(
    "Optimizer:",
    inspect.signature(
        optimization_module.run_optimization
    ),
)

print(
    "Trainer:",
    inspect.signature(
        training_module.train_stacking_regressor
    ),
)


/home/wanderson/.cache/pypoetry/virtualenvs/energy-mlops-G3NdcX_6-py3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Módulos recarregados.
Optimizer: (df_train: pandas.DataFrame, train_file: str, n_trials: int = 20, n_splits: int = 3, stacking_n_splits: int = 5, enabled_estimators: tuple[str, ...] = ('lgbm', 'xgboost', 'rf')) -> energy_mlops.models.interfaces.OptimizationResult
Trainer: (df_train: pandas.DataFrame, df_test: pandas.DataFrame, train_file: str, test_file: str, best_params: dict | None = None, optimization_run_id: str | None = None, enabled_estimators: tuple[str, ...] = ('lgbm', 'xgboost', 'rf')) -> tuple[str, float]


## 4. Configuração do benchmark

A arquitetura permanece fixa. O único fator experimental é a janela.

`RUN_EXPERIMENTS=False` impede iniciar três otimizações longas por
acidente ao abrir ou executar o notebook.

`PERSIST_DERIVED_DATASETS=False` impede escrita acidental dos artefatos
TRAIN/OOT no Gold.


In [2]:
EXPERIMENT_CONFIG = {
    "n_trials": 20,
    "n_splits": 3,
    "stacking_n_splits": 5,
}

TRAINING_WINDOWS = {
    "Expanding": None,
    "Rolling 24m": 24,
    "Rolling 12m": 12,
}

ENABLED_ESTIMATORS = (
    validate_enabled_estimators(
        (
            "lgbm",
            "xgboost",
            "rf",
        )
    )
)

RUN_EXPERIMENTS = False
PERSIST_DERIVED_DATASETS = False

EXPERIMENT_CONFIG, TRAINING_WINDOWS, ENABLED_ESTIMATORS


({'n_trials': 20, 'n_splits': 3, 'stacking_n_splits': 5},
 {'Expanding': None, 'Rolling 24m': 24, 'Rolling 12m': 12},
 ('lgbm', 'xgboost', 'rf'))

## 5. Snapshot temporal congelado e proveniência

O benchmark utiliza o mesmo snapshot Gold validado anteriormente:

```text
GOLD
arquivo: dataset_renewable_energy_2024_03_2026_09.parquet
rows: 22,145
range: 2024-03-21 00:00:00 → 2026-10-01 02:00:00
missing timestamps: 34
gaps: 2
features: 9

OOT — setembro/2026
rows: 720
range: 2026-09-01 03:00:00 → 2026-10-01 02:00:00
missing timestamps: 0
gaps: 0
```

A capacidade instalada segue o contrato canônico
ABEEólica/INFOVENTO. O OOT é mantido idêntico em todos os tratamentos.


In [3]:
GOLD_FILE = (
    "dataset_renewable_energy_2024_03_2026_09.parquet"
)

GOLD_PATH = (
    f"s3://{settings.RUSTFS_BUCKET}/gold/"
    f"{GOLD_FILE}"
)

OOT_YEAR = 2026
OOT_MONTH = 9

EXPECTED_GOLD_ROWS = 22_145
EXPECTED_OOT_ROWS = 720

EXPECTED_GOLD_START = pd.Timestamp(
    "2024-03-21 00:00:00"
)
EXPECTED_GOLD_END = pd.Timestamp(
    "2026-10-01 02:00:00"
)
EXPECTED_TRAIN_END = pd.Timestamp(
    "2026-09-01 02:00:00"
)
EXPECTED_OOT_START = pd.Timestamp(
    "2026-09-01 03:00:00"
)
EXPECTED_OOT_END = pd.Timestamp(
    "2026-10-01 02:00:00"
)

EXPECTED_WINDOWS = {
    "Expanding": {
        "months": None,
        "rows": 21_425,
        "start": pd.Timestamp(
            "2024-03-21 00:00:00"
        ),
        "missing_timestamps": 34,
        "gaps": 2,
    },
    "Rolling 24m": {
        "months": 24,
        "rows": 17_486,
        "start": pd.Timestamp(
            "2024-09-01 03:00:00"
        ),
        "missing_timestamps": 34,
        "gaps": 2,
    },
    "Rolling 12m": {
        "months": 12,
        "rows": 8_760,
        "start": pd.Timestamp(
            "2025-09-01 03:00:00"
        ),
        "missing_timestamps": 0,
        "gaps": 0,
    },
}


In [4]:
df_gold = pd.read_parquet(
    GOLD_PATH,
    storage_options=settings.storage_options,
)

gold_audit = audit_gold_snapshot(
    df_gold
)

MODEL_FEATURES = (
    get_model_feature_columns()
)

window_datasets = {}

for label, months in TRAINING_WINDOWS.items():
    df_train, df_oot = (
        split_training_window_snapshot(
            df_gold,
            oot_year=OOT_YEAR,
            oot_month=OOT_MONTH,
            training_window_months=months,
        )
    )

    train_audit = audit_gold_snapshot(
        df_train
    )
    oot_audit = audit_gold_snapshot(
        df_oot
    )

    train_file, oot_file = (
        build_training_dataset_labels(
            df_train,
            oot_year=OOT_YEAR,
            oot_month=OOT_MONTH,
            training_window_months=months,
        )
    )

    window_datasets[label] = {
        "months": months,
        "train": df_train,
        "oot": df_oot,
        "train_audit": train_audit,
        "oot_audit": oot_audit,
        "train_file": train_file,
        "oot_file": oot_file,
    }

print("✅ Janelas derivadas do snapshot Gold.")


✅ Janelas derivadas do snapshot Gold.


In [5]:
snapshot_rows = []

for label, payload in window_datasets.items():
    df_train = payload["train"]
    df_oot = payload["oot"]

    snapshot_rows.append(
        {
            "Janela": label,
            "Meses": payload["months"],
            "Train rows": len(df_train),
            "Train início": df_train["date"].min(),
            "Train fim": df_train["date"].max(),
            "Train gaps": (
                payload["train_audit"].gap_count
            ),
            "Train missing": (
                payload[
                    "train_audit"
                ].missing_timestamps
            ),
            "OOT rows": len(df_oot),
            "OOT início": df_oot["date"].min(),
            "OOT fim": df_oot["date"].max(),
            "TRAIN file": payload["train_file"],
            "OOT file": payload["oot_file"],
        }
    )

snapshot_summary = (
    pd.DataFrame(snapshot_rows)
    .set_index("Janela")
)

snapshot_summary


,Meses,Train rows,Train início,Train fim,Train gaps,Train missing,OOT rows,OOT início,OOT fim,TRAIN file,OOT file
Janela,,,,,,,,,,,
Expanding,NaN,21425,2024-03-21 00:00:00,2026-09-01 02:00:00,2,34,720,2026-09-01 03:00:00,2026-10-01 02:00:00,train_wind_energy_2024_03_expanding_up_to_2026...,oot_test_wind_energy_2026_09.parquet
Rolling 24m,24.0,17486,2024-09-01 03:00:00,2026-09-01 02:00:00,2,34,720,2026-09-01 03:00:00,2026-10-01 02:00:00,train_wind_energy_2024_09_rolling_24m_up_to_20...,oot_test_wind_energy_2026_09.parquet
Rolling 12m,12.0,8760,2025-09-01 03:00:00,2026-09-01 02:00:00,0,0,720,2026-09-01 03:00:00,2026-10-01 02:00:00,train_wind_energy_2025_09_rolling_12m_up_to_20...,oot_test_wind_energy_2026_09.parquet


## 6. Sanity checks antes da execução

O benchmark só deve rodar se:

1. o Gold permanecer exatamente no snapshot congelado;
2. as três janelas tiverem os limites temporais esperados;
3. o OOT for exatamente o mesmo DataFrame nos três tratamentos;
4. TRAIN e OOT não tiverem overlap;
5. o OOT estiver completo e sem gaps;
6. os gaps históricos permanecerem somente nas janelas que os incluem;
7. o contrato do modelo continuar com exatamente 9 features;
8. a arquitetura permanecer `LGBM + XGB + RF`;
9. o orçamento Optuna permanecer idêntico;
10. nenhuma execução utilizar o OOT durante a otimização.


In [6]:
def validate_experiment_inputs():
    assert len(df_gold) == EXPECTED_GOLD_ROWS

    assert (
        pd.Timestamp(
            df_gold["date"].min()
        )
        == EXPECTED_GOLD_START
    )

    assert (
        pd.Timestamp(
            df_gold["date"].max()
        )
        == EXPECTED_GOLD_END
    )

    assert gold_audit.missing_timestamps == 34
    assert gold_audit.gap_count == 2

    assert len(MODEL_FEATURES) == 9

    assert ENABLED_ESTIMATORS == (
        "lgbm",
        "xgboost",
        "rf",
    )

    assert EXPERIMENT_CONFIG == {
        "n_trials": 20,
        "n_splits": 3,
        "stacking_n_splits": 5,
    }

    required_columns = {
        "target_fc",
        "wind_generation_mw",
        "capacidade_mw",
        *MODEL_FEATURES,
    }

    reference_oot = None

    for label, expected in (
        EXPECTED_WINDOWS.items()
    ):
        payload = window_datasets[label]

        assert (
            payload["months"]
            == expected["months"]
        )

        df_train = payload["train"]
        df_oot = payload["oot"]

        train_audit = payload[
            "train_audit"
        ]
        oot_audit = payload[
            "oot_audit"
        ]

        assert len(df_train) == expected["rows"]

        assert (
            pd.Timestamp(
                df_train["date"].min()
            )
            == expected["start"]
        )

        assert (
            pd.Timestamp(
                df_train["date"].max()
            )
            == EXPECTED_TRAIN_END
        )

        assert len(df_oot) == EXPECTED_OOT_ROWS

        assert (
            pd.Timestamp(
                df_oot["date"].min()
            )
            == EXPECTED_OOT_START
        )

        assert (
            pd.Timestamp(
                df_oot["date"].max()
            )
            == EXPECTED_OOT_END
        )

        assert (
            df_train["date"].max()
            < df_oot["date"].min()
        )

        assert (
            train_audit.missing_timestamps
            == expected[
                "missing_timestamps"
            ]
        )

        assert (
            train_audit.gap_count
            == expected["gaps"]
        )

        assert (
            oot_audit.missing_timestamps
            == 0
        )
        assert oot_audit.gap_count == 0

        missing_train = (
            required_columns
            - set(df_train.columns)
        )
        assert not missing_train, (
            f"{label}: colunas ausentes no TRAIN: "
            f"{sorted(missing_train)}"
        )

        missing_oot = (
            required_columns
            - set(df_oot.columns)
        )
        assert not missing_oot, (
            f"{label}: colunas ausentes no OOT: "
            f"{sorted(missing_oot)}"
        )

        if reference_oot is None:
            reference_oot = df_oot
        else:
            pd.testing.assert_frame_equal(
                reference_oot,
                df_oot,
            )

    print("✅ Snapshot Gold validado.")
    print("✅ Expanding / 24m / 12m validadas.")
    print("✅ OOT idêntico nos três tratamentos.")
    print("✅ TRAIN/OOT sem overlap.")
    print("✅ Contrato de 9 features preservado.")
    print("✅ Arquitetura LGBM + XGB + RF congelada.")
    print("✅ Benchmark pronto para execução.")


validate_experiment_inputs()


✅ Snapshot Gold validado.
✅ Expanding / 24m / 12m validadas.
✅ OOT idêntico nos três tratamentos.
✅ TRAIN/OOT sem overlap.
✅ Contrato de 9 features preservado.
✅ Arquitetura LGBM + XGB + RF congelada.
✅ Benchmark pronto para execução.


## 7. Persistência opcional dos artefatos derivados

O optimizer e o trainer registram no MLflow os nomes dos arquivos
utilizados na linhagem.

Antes da primeira execução, os três TRAIN derivados podem ser
materializados no Gold. O arquivo OOT possui o mesmo conteúdo e o mesmo
nome nos três tratamentos.

A flag permanece `False` por segurança.


In [ ]:
if PERSIST_DERIVED_DATASETS:
    for label, payload in (
        window_datasets.items()
    ):
        (
            persisted_train_file,
            persisted_oot_file,
        ) = persist_training_datasets_task.fn(
            payload["train"],
            payload["oot"],
            oot_year=OOT_YEAR,
            oot_month=OOT_MONTH,
            training_window_months=(
                payload["months"]
            ),
        )

        assert (
            persisted_train_file
            == payload["train_file"]
        )

        assert (
            persisted_oot_file
            == payload["oot_file"]
        )

        print(
            f"✅ {label}: "
            f"{persisted_train_file}"
        )

    print(
        "✅ Artefatos derivados persistidos."
    )
else:
    print(
        "ℹ️ Persistência desabilitada. "
        "Defina PERSIST_DERIVED_DATASETS=True "
        "somente quando quiser materializar "
        "os artefatos derivados."
    )


## 8. Governança do benchmark

Este notebook chama diretamente:

```text
run_optimization()
        ↓
train_stacking_regressor()
        ↓
MLflow
```

Ele **não chama**:

```text
continuous_training_pipeline()
evaluate_and_promote()
```

Portanto, nenhuma run experimental pode alterar automaticamente o
alias `@champion`.

Dependendo do comportamento atual do trainer, versões experimentais
podem ainda ser registradas no Model Registry. Isso é aceitável para
rastreabilidade desde que nenhuma promoção automática seja executada.

As runs recebem tags explícitas com a janela utilizada.


In [ ]:
mlflow_client = MlflowClient(
    tracking_uri=settings.MLFLOW_TRACKING_URI
)


def tag_window_run(
    run_id: str,
    *,
    label: str,
    months: int | None,
    df_train: pd.DataFrame,
    df_oot: pd.DataFrame,
    run_role: str,
):
    strategy = (
        "expanding"
        if months is None
        else "rolling"
    )

    tags = {
        "experiment_family": (
            "training_window_comparison"
        ),
        "experiment_variable": (
            "training_window"
        ),
        "benchmark_label": label,
        "run_role": run_role,
        "training_window_strategy": strategy,
        "training_window_months": (
            "all_available"
            if months is None
            else str(months)
        ),
        "training_start": str(
            df_train["date"].min()
        ),
        "training_end": str(
            df_train["date"].max()
        ),
        "oot_start": str(
            df_oot["date"].min()
        ),
        "oot_end": str(
            df_oot["date"].max()
        ),
        "fixed_architecture": (
            "lgbm,xgboost,rf"
        ),
        "quality_gate_executed": "false",
    }

    for key, value in tags.items():
        mlflow_client.set_tag(
            run_id,
            key,
            value,
        )


## 9. Execução — Expanding vs Rolling 24m vs Rolling 12m

Cada tratamento recebe uma otimização Optuna independente porque a
distribuição de treino muda com a janela.

A arquitetura e o orçamento permanecem fixos.

A flag `RUN_EXPERIMENTS` deve ser alterada para `True` somente quando
o benchmark estiver pronto para ser executado.


In [ ]:
experiment_runs = {}

if RUN_EXPERIMENTS:
    validate_experiment_inputs()

    for label, payload in (
        window_datasets.items()
    ):
        print(
            "\n"
            + "=" * 72
        )
        print(
            f"🚀 Executando: {label}"
        )
        print(
            "=" * 72
        )

        optimization_result = (
            optimization_module.run_optimization(
                df_train=payload["train"],
                train_file=(
                    payload["train_file"]
                ),
                n_trials=(
                    EXPERIMENT_CONFIG[
                        "n_trials"
                    ]
                ),
                n_splits=(
                    EXPERIMENT_CONFIG[
                        "n_splits"
                    ]
                ),
                stacking_n_splits=(
                    EXPERIMENT_CONFIG[
                        "stacking_n_splits"
                    ]
                ),
                enabled_estimators=(
                    ENABLED_ESTIMATORS
                ),
            )
        )

        optimization_run_id = (
            optimization_result[
                "optimization_run_id"
            ]
        )

        tag_window_run(
            optimization_run_id,
            label=label,
            months=payload["months"],
            df_train=payload["train"],
            df_oot=payload["oot"],
            run_role="optimization",
        )

        (
            training_run_id,
            oot_mae_mw,
        ) = (
            training_module
            .train_stacking_regressor(
                df_train=payload["train"],
                df_test=payload["oot"],
                train_file=(
                    payload["train_file"]
                ),
                test_file=(
                    payload["oot_file"]
                ),
                best_params=(
                    optimization_result[
                        "best_params"
                    ]
                ),
                optimization_run_id=(
                    optimization_run_id
                ),
                enabled_estimators=(
                    ENABLED_ESTIMATORS
                ),
            )
        )

        tag_window_run(
            training_run_id,
            label=label,
            months=payload["months"],
            df_train=payload["train"],
            df_oot=payload["oot"],
            run_role="training",
        )

        experiment_runs[label] = {
            "optimization_run_id": (
                optimization_run_id
            ),
            "training_run_id": (
                training_run_id
            ),
            "oot_mae_mw": (
                oot_mae_mw
            ),
        }

experiment_runs


## 10. Congelamento dos Run IDs

Após a primeira execução bem-sucedida:

1. copie os três `training_run_id` produzidos acima;
2. preencha `FROZEN_TRAINING_RUN_IDS`;
3. retorne `RUN_EXPERIMENTS=False`;
4. mantenha os IDs congelados no notebook.

Isso permite reconstruir a tabela de resultados sem depender do estado
atual do kernel nem da ordenação das runs no MLflow.


In [ ]:
FROZEN_TRAINING_RUN_IDS = {
    "Expanding": None,
    "Rolling 24m": None,
    "Rolling 12m": None,
}

if experiment_runs:
    current_training_run_ids = {
        label: payload[
            "training_run_id"
        ]
        for label, payload in (
            experiment_runs.items()
        )
    }

    print(
        "IDs produzidos nesta execução:"
    )
    print(
        json.dumps(
            current_training_run_ids,
            indent=2,
        )
    )
else:
    print(
        "ℹ️ Nenhuma execução realizada "
        "nesta sessão."
    )

FROZEN_TRAINING_RUN_IDS


## 11. Resultados consolidados

A tabela abaixo recupera diretamente das runs MLflow:

- CV nMAE médio e desvio;
- OOT MAE em MW;
- OOT nMAE;
- OOT MAE em fator de capacidade;
- OOT R²;
- pesos OOF do meta-learner;
- duração da otimização;
- duração do treinamento;
- limites efetivos do TRAIN.

Nenhuma equivalência estatística é inferida apenas a partir das
diferenças pontuais.


In [ ]:
def run_duration_minutes(run):
    start_time = run.info.start_time
    end_time = run.info.end_time

    if (
        start_time is None
        or end_time is None
    ):
        return None

    return (
        end_time - start_time
    ) / 60_000


def resolve_training_run_ids():
    if experiment_runs:
        return {
            label: payload[
                "training_run_id"
            ]
            for label, payload in (
                experiment_runs.items()
            )
        }

    return (
        FROZEN_TRAINING_RUN_IDS.copy()
    )


training_run_ids = (
    resolve_training_run_ids()
)

missing_run_ids = [
    label
    for label, run_id in (
        training_run_ids.items()
    )
    if not run_id
]

if missing_run_ids:
    benchmark_results = None

    print(
        "ℹ️ Resultados ainda não congelados."
    )
    print(
        "Preencha os Run IDs para:",
        missing_run_ids,
    )

else:
    benchmark_rows = []

    for label, training_run_id in (
        training_run_ids.items()
    ):
        training_run = (
            mlflow_client.get_run(
                training_run_id
            )
        )

        optimization_run_id = (
            training_run.data.tags[
                "optimization_run_id"
            ]
        )

        optimization_run = (
            mlflow_client.get_run(
                optimization_run_id
            )
        )

        meta_weights = json.loads(
            training_run.data.tags.get(
                "meta_weights_json",
                "{}",
            )
        )

        payload = window_datasets[
            label
        ]

        benchmark_rows.append(
            {
                "Janela": label,
                "Train rows": len(
                    payload["train"]
                ),
                "Train início": (
                    payload[
                        "train"
                    ]["date"].min()
                ),
                "Train fim": (
                    payload[
                        "train"
                    ]["date"].max()
                ),
                "CV nMAE mean (%)": (
                    optimization_run
                    .data.metrics.get(
                        "best_cv_nmae_pct_mean"
                    )
                ),
                "CV nMAE std (%)": (
                    optimization_run
                    .data.metrics.get(
                        "best_cv_nmae_pct_std"
                    )
                ),
                "OOT MAE (MW)": (
                    training_run
                    .data.metrics.get(
                        "oot_mae_mw"
                    )
                ),
                "OOT nMAE (%)": (
                    training_run
                    .data.metrics.get(
                        "oot_nmae_pct"
                    )
                ),
                "OOT MAE FC (%)": (
                    training_run
                    .data.metrics.get(
                        "oot_mae_fc_pct"
                    )
                ),
                "OOT R²": (
                    training_run
                    .data.metrics.get(
                        "oot_r2_score"
                    )
                ),
                "Peso LGBM": (
                    meta_weights.get(
                        "lgbm"
                    )
                ),
                "Peso XGB": (
                    meta_weights.get(
                        "xgboost"
                    )
                ),
                "Peso RF": (
                    meta_weights.get(
                        "rf"
                    )
                ),
                "Optuna (min)": (
                    run_duration_minutes(
                        optimization_run
                    )
                ),
                "Treino (min)": (
                    run_duration_minutes(
                        training_run
                    )
                ),
                "Optimization Run ID": (
                    optimization_run_id
                ),
                "Training Run ID": (
                    training_run_id
                ),
            }
        )

    benchmark_results = (
        pd.DataFrame(
            benchmark_rows
        )
        .set_index("Janela")
    )

    display(
        benchmark_results.round(4)
    )


## 12. Diferenças em relação à expanding window

A expanding window é utilizada como baseline porque representa a
política anterior do projeto.

As diferenças abaixo são descritivas:

```text
Δ MAE  > 0  → maior erro que expanding
Δ nMAE > 0  → maior erro normalizado que expanding
Δ R²   < 0  → menor R² que expanding
```

Essas diferenças não constituem, por si só, teste de equivalência,
não-inferioridade ou significância estatística.


In [ ]:
if benchmark_results is None:
    window_deltas = None

    print(
        "ℹ️ Execute ou congele as runs "
        "antes de calcular os deltas."
    )

else:
    baseline = (
        benchmark_results.loc[
            "Expanding"
        ]
    )

    window_deltas = (
        benchmark_results.loc[
            [
                "Rolling 24m",
                "Rolling 12m",
            ],
            [
                "Train rows",
                "CV nMAE mean (%)",
                "CV nMAE std (%)",
                "OOT MAE (MW)",
                "OOT nMAE (%)",
                "OOT R²",
                "Optuna (min)",
                "Treino (min)",
            ],
        ]
        .copy()
    )

    window_deltas[
        "Δ MAE vs Expanding (MW)"
    ] = (
        window_deltas[
            "OOT MAE (MW)"
        ]
        - baseline[
            "OOT MAE (MW)"
        ]
    )

    window_deltas[
        "Δ nMAE vs Expanding (p.p.)"
    ] = (
        window_deltas[
            "OOT nMAE (%)"
        ]
        - baseline[
            "OOT nMAE (%)"
        ]
    )

    window_deltas[
        "Δ R² vs Expanding"
    ] = (
        window_deltas[
            "OOT R²"
        ]
        - baseline[
            "OOT R²"
        ]
    )

    display(
        window_deltas.round(4)
    )


## 13. Guia de interpretação

Ao concluir o benchmark, a análise deve considerar em conjunto:

### Generalização OOT
- MAE MW;
- nMAE;
- MAE em fator de capacidade;
- R².

### Estabilidade interna
- CV nMAE mean;
- CV nMAE std.

### Comportamento do ensemble
- pesos LGBM/XGB/RF;
- eventual mudança relevante na composição do meta-learner.

### Custo
- número de observações de treino;
- tempo de Optuna;
- tempo de treinamento.

Exemplos de interpretações possíveis:

```text
Rolling 12m melhora OOT,
mas apresenta CV mais instável.

Rolling 24m preserva OOT
com menor custo de treinamento.

Expanding continua superior,
sugerindo benefício do histórico longo.
```

A conclusão deve ser baseada nos resultados observados. Não assumir
antecipadamente que dados mais recentes ou mais históricos são melhores.


## 14. Quality Gate e isolamento operacional

Este benchmark é deliberadamente separado da promoção automática.

O Quality Gate atual do projeto avalia Champion vs Challenger no
`continuous_training_pipeline()`. Esse mecanismo não é apropriado para
executar três tratamentos experimentais em sequência, pois cada run
poderia produzir uma decisão operacional antes de a comparação estar
concluída.

Neste notebook:

```text
Quality Gate executado: NÃO
Champion alterado automaticamente: NÃO
Alias @champion alterado: NÃO
```

Após a análise, uma eventual mudança da política de janela deve ser
incorporada ao fluxo operacional em um commit separado e submetida aos
testes e ao CI/CD.


## 15. Conclusão — preencher após a execução

Registrar aqui:

- qual janela apresentou o menor OOT MAE/nMAE;
- se a diferença em relação à expanding window é relevante do ponto de
  vista operacional;
- como o CV se comportou em cada janela;
- se os pesos do meta-learner mudaram materialmente;
- impacto sobre custo de otimização/treinamento;
- limitações do benchmark.

Evitar afirmar equivalência estatística sem um teste específico.

### Decisão experimental

```text
Expanding:
[preencher]

Rolling 24m:
[preencher]

Rolling 12m:
[preencher]

Política candidata para validação operacional:
[preencher]
```


## 16. Próximos passos

Após a execução e interpretação:

1. congelar os Run IDs MLflow no notebook;
2. documentar os resultados no notebook técnico principal;
3. atualizar o Model Card somente se a evidência alterar a política
   recomendada de treinamento;
4. decidir se a política operacional deve permanecer expanding ou
   migrar para rolling;
5. caso haja mudança operacional, implementá-la separadamente e passar
   novamente por testes, `tox` e CI/CD;
6. manter estudos futuros de drift e seleção adaptativa de janela como
   questões separadas — este benchmark compara apenas três políticas
   temporais pré-definidas.
